# slice the single wav basing on window automatically
---

In [ ]:
"""
This script provides functionality for cropping audio files. It includes the following imports:
The script is intended to be run in a Jupyter notebook environment and uses interactive widgets to facilitate user interaction.
"""

import os
from pathlib import Path
import glob

import matplotlib.pyplot as plt
import numpy as np

from pydub import AudioSegment
import pydub

In [2]:
def draw_single_slice_range(original_data, fr, slice_idx_list, file_name):
    """
    Plots the waveform of the original data and overlays slice ranges as colored boxes.
    Parameters:
    -----------
    original_data : numpy.ndarray
        The audio data to be plotted.
    fr : int or float
        The sampling frequency of the audio data.
    slice_idx_list : list of tuples
        A list of slice index ranges, where each tuple contains two integers 
        representing the start and end indices of a slice. Alternating slices 
        are colored green and red.
    file_name : str
        The title of the plot, typically the name of the file being visualized.
    """
    time = np.arange(0, len(original_data)) / fr  # Create a time array
    plt.figure(figsize=(20, 10))
    plot_a = plt.subplot(211)
    plot_a.plot(time, original_data)
    plot_a.set_ylabel("Amplitude")
    plot_a.set_xlim(0, len(original_data) / fr)
    plot_a.set_title(str(file_name))
    # draw the slice boxes #
    max_data = original_data.max()

    for i, v in enumerate(slice_idx_list):
        if i % 2 == 1:
            plot_a.vlines(v[0] / 1000, -max_data - 5000, max_data + 5000, color="green")
            plot_a.vlines(v[1] / 1000, -max_data - 5000, max_data + 5000, color="green")
            plot_a.hlines(-max_data - 5000, v[0] / 1000, v[1] / 1000, color="green")
            plot_a.hlines(max_data + 5000, v[0] / 1000, v[1] / 1000, color="green")
        else:
            plot_a.vlines(v[0] / 1000, -max_data - 5000, max_data + 5000, color="red")
            plot_a.vlines(v[1] / 1000, -max_data - 5000, max_data + 5000, color="red")
            plot_a.hlines(-max_data - 5000, v[0] / 1000, v[1] / 1000, color="red")
            plot_a.hlines(max_data + 5000, v[0] / 1000, v[1] / 1000, color="red")


def save_single_slice_range(ori_audio, slice_idx_list, sliced_audio_path):
    """
    Save a specific range of an audio file as a new audio file.
    This function takes an original audio object, extracts a specific slice
    of the audio based on the provided index range, and saves the sliced
    audio to the specified file path in WAV format.
    Args:
        ori_audio: An audio object representing the original audio data.
        slice_idx_list (list of tuples): A list containing tuples where each
            tuple specifies the start and end indices of the slice to extract.
            Only the first tuple in the list is used.
        sliced_audio_path (str): The file path where the sliced audio will be
            saved in WAV format.
    """
    ori_audio = ori_audio[slice_idx_list[0][0]:slice_idx_list[0][1]]

    ori_audio.export(sliced_audio_path, format="wav")


def get_single_slice_range(audio_path, min_silence_len, silence_thresh, user_set_window, output_save_path, pad_to_user_window=True):
    """
    Get the single slice range of the audio file.
    """
    # Create the output directory if it doesn't exist
    output_save_path.mkdir(exist_ok=True)

    # Load the audio file
    audio_segment = AudioSegment.from_file(audio_path, format="wav")
    channel_count = audio_segment.channels
    if channel_count > 1:
        audio_segment = audio_segment.set_channels(1)  # Convert to mono
    frames_per_second = audio_segment.frame_rate
    if frames_per_second != 16000:
        audio_segment = audio_segment.set_frame_rate(16000)
    sample_width = audio_segment.sample_width
    if sample_width != 2:
        audio_segment = audio_segment.set_sample_width(2)  # Convert to 16-bit
    print(f"channel_num: {channel_count}, frame_rate: {frames_per_second}, length: {(len(audio_segment) / 1000.0)}, sample_width: {sample_width}")

    # assert len(audio_segment) >= user_set_window, f"user set window should be smaller than the size of wav file."
    if len(audio_segment) > user_set_window:

        not_silence_ranges = pydub.silence.detect_nonsilent(audio_segment, min_silence_len=min_silence_len, silence_thresh=silence_thresh, seek_step=1)
        print(f"original slice range (ms): {not_silence_ranges}")

        # Combine the ranges list's first value with the last value
        single_slice_range = []
        if len(not_silence_ranges) > 1:
            single_slice_range = (not_silence_ranges[0][0], not_silence_ranges[-1][1])
        elif len(not_silence_ranges) == 1:
            single_slice_range = not_silence_ranges[0]
        else:
            print("No slice range found")

        # update the not_silence_ranges to be more accurate & basing on user setting
        slice_window = single_slice_range[1] - single_slice_range[0]
        if slice_window > user_set_window:
            # if the slice window is too long, we need to split it into two, first from begin, second from end
            not_silence_ranges = [(single_slice_range[0], single_slice_range[0] + user_set_window), (single_slice_range[1] - user_set_window, single_slice_range[1])]
            not_silence_ranges = [not_silence_ranges[0]]
        else:
            # if the slice window is too short, we need to pad it before&after
            pad_size = np.ceil((user_set_window - slice_window) / 2)
            pad_idx_begin = 0
            pad_idx_end = 0

            # pad before
            if single_slice_range[0] - pad_size > 0:
                pad_idx_begin = single_slice_range[0] - pad_size
            else:
                pad_idx_begin = 0
                # update the pad size
                pad_size = (user_set_window - slice_window) - single_slice_range[0]

            # pad after
            if single_slice_range[1] + pad_size < len(audio_segment):
                pad_idx_end = single_slice_range[1] + pad_size
            else:
                pad_idx_end = len(audio_segment)
                # update the begin index
                pad_size = (user_set_window - slice_window) - (len(audio_segment) - single_slice_range[1])
                pad_idx_begin = single_slice_range[0] - pad_size
            # update the not_silence_ranges
            not_silence_ranges = [(pad_idx_begin, pad_idx_end)]

        print(f"updated slice range (ms): {not_silence_ranges}")
        draw_single_slice_range(np.array(audio_segment.get_array_of_samples()), frames_per_second, not_silence_ranges, Path(audio_path).name)
        sliced_audio_path = os.path.join(output_save_path, (Path(audio_path).stem + "_" + str(user_set_window / 1000) + "s" + "_" + "0" + "_nohash" + Path(audio_path).suffix))
        save_single_slice_range(audio_segment, not_silence_ranges, sliced_audio_path)
    else:
        if pad_to_user_window:
            padding_needed = np.ceil((user_set_window - len(audio_segment)) / 2)
            silence = AudioSegment.silent(duration=padding_needed)
            # Add silence to the end of the audio
            padded_audio = silence + audio_segment + silence
            not_silence_ranges = [(0, len(padded_audio))]

            print(f"audio length is shorter than user set window, padding!! updated slice range (ms): {not_silence_ranges}")
            draw_single_slice_range(np.array(padded_audio.get_array_of_samples()), frames_per_second, not_silence_ranges, Path(audio_path).name)
            sliced_audio_path = os.path.join(output_save_path, (Path(audio_path).stem + "_" + str(user_set_window / 1000) + "s" + "_" + "0" + "_nohash" + Path(audio_path).suffix))
            save_single_slice_range(padded_audio, not_silence_ranges, sliced_audio_path)
        else:
            print(f"audio length is shorter than user set window, {audio_path} SKIP!")
            not_silence_ranges = None

    return not_silence_ranges

<a id="id-control-intro"></a>
# Control Introduction
---

- `FOLDER_PATH`: 欲切割之目標資料夾. 切割後的資料夾在 `dataset\{user set folder_path}`
- `USER_SET_WINDOW`: 欲設定之固定窗口大小.
- `MIN_SILENCE_LEN`: 持续多少时间可认定为静默
- `SILENCE_THRESH`: 声音大小小于多少时可认定为静默，默认值为-16dBFS
- `PAD_TO_USER_WINDOW`: True or False. 如果欲切割wav檔長度小於設定之固定窗口大小，決定是否在原檔前後padding
---
- 建議: 如果自動分割效果不太好，再自行調整`min_silence_len`和`silence_thresh`

# Start the auto slicing

In [ ]:
FOLDER_PATH = "唤醒词-连接成功"
USER_SET_WINDOW = 2000  # (ms)
MIN_SILENCE_LEN = 300
SILENCE_THRESH = -35
PAD_TO_USER_WINDOW = True
AUTOSLICE_OUTPUT_PATH = Path(os.getcwd()) / "dataset" / Path(FOLDER_PATH)

for wav_file in glob.glob(os.path.join(FOLDER_PATH, "*.wav")):
    print(wav_file)
    win_idxs = get_single_slice_range(wav_file, MIN_SILENCE_LEN, SILENCE_THRESH, USER_SET_WINDOW, AUTOSLICE_OUTPUT_PATH)

## Test a single file

In [ ]:
FOLDER_PATH = "功能词-唤醒30s"
AUTOSLICE_OUTPUT_PATH = Path(os.getcwd()) / "dataset" / Path(FOLDER_PATH)

AUDIO_PATH = "功能词-唤醒30s\\F_0565_Voice_command_438.wav"

get_single_slice_range(AUDIO_PATH, MIN_SILENCE_LEN, SILENCE_THRESH, USER_SET_WINDOW, AUTOSLICE_OUTPUT_PATH)

#### Result

In [61]:
# Load the audio file
audio_segment_test = AudioSegment.from_file(r"C:\CYCHEN38\OpenNuvoton\ML_KWS\ML_audio_aq\dataset\功能词-唤醒30s\F_0565_Voice_command_438.wav", format="wav")
channel = audio_segment_test.channels
if channel > 1:
    audio_segment_test = audio_segment_test.set_channels(1)  # Convert to mono
frames = audio_segment_test.frame_rate
if frames != 16000:
    audio_segment_test = audio_segment_test.set_frame_rate(16000)
sw = audio_segment_test.sample_width
if sw != 2:
    audio_segment_test = audio_segment_test.set_sample_width(2)  # Convert to 16-bit
print(f"channel_num: {channel}, frame_rate: {frames}, length: {(len(audio_segment_test) / 1000.0)}, sample_width: {sw}")
len(audio_segment_test)

channel_num: 1, frame_rate: 16000, length: 2.0, sample_width: 2


2000